# 02 — Model Training, Hyperparameter Tuning & Evaluation

Trains all four model families — Logistic Regression, XGBoost, CatBoost, and Explainable Boosting Machine (EBM) — as baselines, tunes each (`RandomizedSearchCV` for LR, Optuna/TPE for the other three), and evaluates all four on the held-out test set with ROC-AUC, PR-AUC, F1, KS statistic, and Brier score.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

from sklearn.model_selection import train_test_split

from dac.config import CONFIG
from dac.data.loader import load_uci_credit
from dac.features.engineering import build_preprocessor, engineer_uci_credit_features, split_feature_columns
from dac.models.evaluate import compare_models, compute_metrics, plot_evaluation_suite
from dac.models.train import (
    build_catboost_pipeline, build_ebm_pipeline, build_logistic_regression_pipeline,
    build_xgboost_pipeline, compute_balanced_sample_weight, compute_scale_pos_weight, fit,
)
from dac.models.tune import tune_catboost, tune_ebm, tune_logistic_regression, tune_xgboost

uci_cfg = CONFIG["data"]["uci_credit"]
df, _ = load_uci_credit()
df = engineer_uci_credit_features(df)

exclude_cols = [uci_cfg["id_col"], *uci_cfg["protected_attributes"]]
numeric_cols, categorical_cols = split_feature_columns(df, uci_cfg["target_col"], exclude_cols)
X = df[numeric_cols + categorical_cols]
y = df[uci_cfg["target_col"]]
sensitive = df[uci_cfg["protected_attributes"]]

X_train, X_test, y_train, y_test, sens_train, sens_test = train_test_split(
    X, y, sensitive, test_size=CONFIG["split"]["test_size"], random_state=CONFIG["seed"], stratify=y
)
preprocessor = build_preprocessor(numeric_cols, categorical_cols)
ebm_weights = compute_balanced_sample_weight(y_train)

## Baselines

In [ ]:
lr_baseline = fit(build_logistic_regression_pipeline(preprocessor), X_train, y_train, "logistic_regression_baseline")
xgb_baseline = fit(
    build_xgboost_pipeline(preprocessor, scale_pos_weight=compute_scale_pos_weight(y_train)),
    X_train, y_train, "xgboost_baseline",
)
cb_baseline = fit(build_catboost_pipeline(preprocessor), X_train, y_train, "catboost_baseline")
ebm_baseline = fit(build_ebm_pipeline(preprocessor), X_train, y_train, "ebm_baseline", sample_weight=ebm_weights)

baseline_results = {}
for trained in (lr_baseline, xgb_baseline, cb_baseline, ebm_baseline):
    proba = trained.pipeline.predict_proba(X_test)[:, 1]
    baseline_results[trained.name] = compute_metrics(y_test.to_numpy(), proba)
    plot_evaluation_suite(y_test.to_numpy(), proba, trained.name, CONFIG["paths"]["figures_dir"] / "uci_credit")
compare_models(baseline_results, CONFIG["paths"]["metrics_dir"] / "baseline")

## Hyperparameter tuning

In [ ]:
lr_tuned = tune_logistic_regression(preprocessor, X_train, y_train, n_iter=15, cv_folds=5, seed=CONFIG["seed"])
xgb_tuned = tune_xgboost(preprocessor, X_train, y_train, n_trials=CONFIG["tuning"]["n_trials"], cv_folds=5, seed=CONFIG["seed"])
cb_tuned = tune_catboost(preprocessor, X_train, y_train, n_trials=CONFIG["tuning"]["catboost_n_trials"], cv_folds=5, seed=CONFIG["seed"])
ebm_tuned = tune_ebm(preprocessor, X_train, y_train, n_trials=CONFIG["tuning"]["ebm_n_trials"], cv_folds=3, seed=CONFIG["seed"])

lr_tuned.pipeline.fit(X_train, y_train)
xgb_tuned.pipeline.fit(X_train, y_train)
cb_tuned.pipeline.fit(X_train, y_train)
ebm_tuned.pipeline.fit(X_train, y_train, clf__sample_weight=ebm_weights)

In [ ]:
tuned_pipelines = {
    "logistic_regression_tuned": lr_tuned.pipeline, "xgboost_tuned": xgb_tuned.pipeline,
    "catboost_tuned": cb_tuned.pipeline, "ebm_tuned": ebm_tuned.pipeline,
}
tuned_results = {}
for name, pipeline in tuned_pipelines.items():
    proba = pipeline.predict_proba(X_test)[:, 1]
    tuned_results[name] = compute_metrics(y_test.to_numpy(), proba)
    plot_evaluation_suite(y_test.to_numpy(), proba, name, CONFIG["paths"]["figures_dir"] / "uci_credit")
compare_models({**baseline_results, **tuned_results}, CONFIG["paths"]["metrics_dir"])
best_model_name = max(tuned_results, key=lambda k: tuned_results[k]["roc_auc"])
print("Best model:", best_model_name)
tuned_results

Continue to `03_explainability_shap.ipynb` for SHAP on XGBoost / the best model, `04_fairness_audit_mitigation.ipynb` for the fairness audit + reweighing mitigation, and `05_ebm_explainability_comparison.ipynb` for EBM's native explanation vs. SHAP.